# 🎙️ Day 1：打造語音助理系統

In [ ]:
%%capture
!pip install openai langchain langchain-openai langgraph

In [ ]:
import os

# 請填入你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

---

## 1. 語音助理概論與應用

### 語音助理的核心架構

一個完整的語音助理系統由以下五大模組組成：

```
🎤 ASR (語音識別) → 🧠 NLU (自然語言理解) → 💬 Dialog (對話管理) → 📝 NLG (自然語言生成) → 🔊 TTS (語音合成)
```

| 模組 | 功能 | 技術範例 |
|------|------|---------|
| **ASR** (Automatic Speech Recognition) | 將語音轉為文字 | OpenAI Whisper, Google Speech-to-Text |
| **NLU** (Natural Language Understanding) | 理解使用者意圖 | LLM (GPT-4.1), BERT |
| **Dialog Management** | 管理對話流程與狀態 | LangGraph, Rasa |
| **NLG** (Natural Language Generation) | 產生自然語言回覆 | LLM (GPT-4.1) |
| **TTS** (Text-to-Speech) | 將文字轉為語音 | OpenAI TTS, Google TTS |

### 現今語音助理的應用場景

- **智慧客服**：銀行、電信、電商的自動語音客服
- **智慧家居**：Alexa、Google Home、Siri
- **醫療輔助**：語音病歷記錄、看診輔助
- **教育學習**：語言學習助教、互動式教學
- **車載系統**：語音導航、車內控制

### 為什麼用 LLM 來打造語音助理？

傳統語音助理需要分別訓練 NLU、Dialog、NLG 三個模組，維護成本高。  
使用 LLM，我們可以將這三個模組合併為一個 LLM 節點，大幅簡化架構：

```
🎤 ASR → 🤖 LLM (NLU + Dialog + NLG) → 🔊 TTS
```

本單元就是基於這個簡化架構來實作。

---

## 2. ASR 語音識別技術

### OpenAI Whisper API

Whisper 是 OpenAI 開發的語音識別模型，支援多種語言（含中文），準確率高。

特點：
- 支援 **98 種語言**，中文表現優秀
- 支援多種音訊格式：mp3, mp4, mpeg, mpga, m4a, wav, webm
- 檔案大小上限：25 MB
- 可指定語言參數提升識別準確度

### Google Colab 錄音工具

在 Colab 上可以直接透過瀏覽器錄音，產生音訊檔供後續使用。

In [ ]:
#@markdown ### 🎤 Colab 瀏覽器錄音工具（執行後點擊錄音按鈕）

from IPython.display import HTML, Audio, display
from base64 import b64decode
import numpy as np

RECORD_HTML = """
<div style="padding:10px; background:#f0f0f0; border-radius:8px; max-width:400px;">
  <h3 style="margin:0 0 10px 0;">🎤 錄音工具</h3>
  <button id="recordBtn" onclick="toggleRecord()" 
          style="padding:10px 24px; font-size:16px; cursor:pointer; border:none; border-radius:6px; background:#4CAF50; color:white;">
    開始錄音
  </button>
  <p id="status" style="margin-top:8px; color:#666;">準備就緒</p>
</div>
<script>
let mediaRecorder, chunks=[], isRecording=false;
async function toggleRecord(){
  const btn=document.getElementById('recordBtn');
  const status=document.getElementById('status');
  if(!isRecording){
    const stream=await navigator.mediaDevices.getUserMedia({audio:true});
    mediaRecorder=new MediaRecorder(stream,{mimeType:'audio/webm'});
    chunks=[];
    mediaRecorder.ondataavailable=e=>chunks.push(e.data);
    mediaRecorder.onstop=async()=>{
      const blob=new Blob(chunks,{type:'audio/webm'});
      const reader=new FileReader();
      reader.onloadend=()=>{
        const base64=reader.result.split(',')[1];
        google.colab.kernel.invokeFunction('notebook.save_audio',[base64],{});
        status.textContent='✅ 錄音已儲存！';
      };
      reader.readAsDataURL(blob);
      stream.getTracks().forEach(t=>t.stop());
    };
    mediaRecorder.start();
    isRecording=true;
    btn.textContent='停止錄音';
    btn.style.background='#f44336';
    status.textContent='🔴 錄音中...';
  }else{
    mediaRecorder.stop();
    isRecording=false;
    btn.textContent='開始錄音';
    btn.style.background='#4CAF50';
    status.textContent='處理中...';
  }
}
</script>
"""

# 定義回呼函數，將錄音資料存為檔案
AUDIO_PATH = "recorded_audio.webm"

def save_audio(base64_data):
    audio_bytes = b64decode(base64_data)
    with open(AUDIO_PATH, "wb") as f:
        f.write(audio_bytes)
    print(f"錄音已儲存至 {AUDIO_PATH}，大小：{len(audio_bytes)} bytes")

# 註冊回呼（僅在 Colab 環境有效）
try:
    from google.colab import output
    output.register_callback("notebook.save_audio", save_audio)
    display(HTML(RECORD_HTML))
except ImportError:
    print("此錄音工具僅支援 Google Colab 環境")
    print("若在本地環境，請直接準備一個音訊檔案（如 sample.wav）")

### 使用 Whisper API 進行語音轉文字

接下來我們用 OpenAI 的 Whisper API 來將音訊檔轉為文字。

In [ ]:
from openai import OpenAI

client = OpenAI()

# --- 先用 TTS 產生一段範例音檔供測試 ---
sample_text = "你好，我想查詢我的帳戶餘額，請問需要提供什麼資訊？"

response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input=sample_text,
)
response.stream_to_file("sample_audio.mp3")
print(f"已產生範例音檔：sample_audio.mp3")

In [ ]:
# --- 使用 Whisper API 進行語音識別 ---
audio_file = open("sample_audio.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file,
    language="zh",  # 指定中文可提升辨識準確度
)

print("原始文字：", sample_text)
print("辨識結果：", transcription.text)

### Whisper API 補充說明

- **語言參數**：指定 `language="zh"` 可避免語言偵測錯誤，提升中文辨識準確度
- **prompt 參數**：可傳入提示詞引導模型，例如特定的專有名詞
- **response_format**：支援 `json`, `text`, `srt`, `verbose_json`, `vtt`
- **限制**：單檔最大 25MB，超過需先切割

```python
# 進階用法：加入 prompt 提示特定術語
transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file,
    language="zh",
    prompt="這是一段關於金融服務的對話，可能包含：定期存款、活期儲蓄、信用卡額度",
)
```

---

## 3. TTS 語音合成技術

### OpenAI TTS API

OpenAI 提供高品質的語音合成服務，支援多種語音風格。

| 模型 | 說明 | 適用場景 |
|------|------|---------|
| `tts-1` | 速度快、延遲低 | 即時對話 |
| `tts-1-hd` | 音質更高 | 預錄內容 |

可用語音：`alloy`, `ash`, `coral`, `echo`, `fable`, `nova`, `onyx`, `sage`, `shimmer`

In [ ]:
# --- TTS 基本使用：文字轉語音 ---
from IPython.display import Audio, display

tts_text = "歡迎來到我們的語音助理課程，今天我們要學習如何打造一個完整的語音問答系統。"

response = client.audio.speech.create(
    model="tts-1",
    voice="nova",
    input=tts_text,
)

# 儲存音檔
response.stream_to_file("tts_output.mp3")
print("語音已產生！")

# 在 notebook 中播放
display(Audio("tts_output.mp3", autoplay=False))

In [ ]:
# --- 比較不同語音風格 ---
test_sentence = "您好，我是您的智慧客服助理，有什麼可以幫您的嗎？"

voices = ["alloy", "echo", "nova", "shimmer"]

for voice in voices:
    print(f"--- 語音：{voice} ---")
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=test_sentence,
    )
    filename = f"voice_{voice}.mp3"
    response.stream_to_file(filename)
    display(Audio(filename, autoplay=False))

---

## 4. 使用 LangGraph 建立簡單客服問答 Agent

### 架構說明

我們要建立一個語音問答 Agent，流程如下：

```
[音訊輸入] → STT 節點 → LLM 節點 → TTS 節點 → [音訊輸出]
```

使用 LangGraph 的 `StateGraph` 來管理狀態與流程。

### Step a) 定義 State

In [ ]:
from typing import Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# --- 定義狀態 ---
class VoiceState(TypedDict):
    audio_path: Optional[str]      # 輸入音訊路徑
    user_text: Optional[str]       # STT 辨識結果
    answer: Optional[str]          # LLM 回覆文字
    audio_output: Optional[str]    # TTS 輸出音訊路徑

### Step b) STT 節點 — 語音轉文字

In [ ]:
# --- STT 節點：語音轉文字 ---
def stt_node(state: VoiceState) -> dict:
    """將音訊檔案轉為文字"""
    audio_file = open(state["audio_path"], "rb")
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        language="zh",
    )
    print(f"[STT] 辨識結果：{transcription.text}")
    return {"user_text": transcription.text}

### Step c) LLM 節點 — 產生回覆

In [ ]:
# --- LLM 節點：產生客服回覆 ---
llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0.3)

SYSTEM_PROMPT = """你是一位專業的銀行客服人員。
請用親切、專業的語氣回答客戶的問題。
回覆請簡潔明瞭，適合語音播放（不要使用 markdown 格式、表情符號或特殊符號）。
如果無法回答，請禮貌地引導客戶撥打客服專線。"""

def llm_node(state: VoiceState) -> dict:
    """使用 LLM 產生客服回覆"""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=state["user_text"]),
    ]
    response = llm.invoke(messages)
    print(f"[LLM] 回覆：{response.content}")
    return {"answer": response.content}

### Step d) TTS 節點 — 文字轉語音

In [ ]:
# --- TTS 節點：文字轉語音 ---
def tts_node(state: VoiceState) -> dict:
    """將回覆文字轉為語音"""
    output_path = "agent_response.mp3"
    response = client.audio.speech.create(
        model="tts-1",
        voice="nova",
        input=state["answer"],
    )
    response.stream_to_file(output_path)
    print(f"[TTS] 語音已儲存至 {output_path}")
    return {"audio_output": output_path}

### Step e) 組裝 Graph：STT → LLM → TTS → END

In [ ]:
# --- 建立 LangGraph 流程 ---
graph = StateGraph(VoiceState)

# 加入節點
graph.add_node("stt", stt_node)
graph.add_node("llm", llm_node)
graph.add_node("tts", tts_node)

# 定義流程：STT → LLM → TTS → END
graph.add_edge(START, "stt")
graph.add_edge("stt", "llm")
graph.add_edge("llm", "tts")
graph.add_edge("tts", END)

# 編譯
voice_agent = graph.compile()

print("語音助理 Agent 建立完成！")

In [ ]:
# 視覺化 Graph 結構
from IPython.display import Image, display

display(Image(voice_agent.get_graph().draw_mermaid_png()))

### Step f) 執行語音助理

使用前面產生的範例音檔來測試完整流程。

In [ ]:
# --- 執行語音助理 ---
result = voice_agent.invoke({
    "audio_path": "sample_audio.mp3",
})

print("\n===== 完整結果 =====")
print(f"使用者說：{result['user_text']}")
print(f"助理回覆：{result['answer']}")
print(f"語音檔案：{result['audio_output']}")

# 播放助理回覆的語音
display(Audio(result["audio_output"], autoplay=False))

In [ ]:
# ============================
# 實作練習模板
# ============================
# 請修改以下三個部分，建立你自己的領域語音助理

# 1. 修改 System Prompt（改成你選擇的領域）
MY_SYSTEM_PROMPT = """你是一位專業的 _____ 服務人員。
請用親切、專業的語氣回答客戶的問題。
回覆請簡潔明瞭，適合語音播放。
"""

# 2. 修改 Normalize Prompt（加入你的領域術語）
MY_NORMALIZE_PROMPT = """你是一位 _____ 領域的文字校正專家。
使用者的文字來自語音辨識，可能有錯別字或術語辨識錯誤。

請修正以下常見的術語錯誤：
- XXX → OOO
- XXX → OOO

只修正明顯的辨識錯誤，不要改變語意。只輸出修正後的文字。"""

# 3. 定義你的節點函數
def my_normalize_node(state: VoiceState) -> dict:
    """修正語音辨識中的領域術語錯誤"""
    messages = [
        SystemMessage(content=MY_NORMALIZE_PROMPT),
        HumanMessage(content=state["user_text"]),
    ]
    response = llm.invoke(messages)
    corrected = response.content.strip()
    print(f"[Normalize] {state['user_text']} → {corrected}")
    return {"user_text": corrected}

def my_llm_node(state: VoiceState) -> dict:
    """使用 LLM 產生回覆"""
    messages = [
        SystemMessage(content=MY_SYSTEM_PROMPT),
        HumanMessage(content=state["user_text"]),
    ]
    response = llm.invoke(messages)
    print(f"[LLM] {response.content}")
    return {"answer": response.content}

# 4. 組裝你的 Graph
my_graph = StateGraph(VoiceState)
my_graph.add_node("stt", stt_node)
my_graph.add_node("normalize", my_normalize_node)
my_graph.add_node("llm", my_llm_node)
my_graph.add_node("tts", tts_node)

my_graph.add_edge(START, "stt")
my_graph.add_edge("stt", "normalize")
my_graph.add_edge("normalize", "llm")
my_graph.add_edge("llm", "tts")
my_graph.add_edge("tts", END)

my_agent = my_graph.compile()
print("你的語音助理已建立！")
